# 11 — Session-Independent Generalization

## Objective

This notebook evaluates whether transformation-based synthetic augmentation improves generalization to previously unseen radar recording sessions.

Unlike the official segment-level split, the session-independent split assigns every segment from a recording session to exactly one partition:

- training;
- validation;
- test.

This prevents segments from the same acquisition session from appearing in multiple partitions.

## Experimental policy

1. Construct the split using session identifiers only.
2. Preserve bird/drone representation across partitions.
3. Exclude edge-flagged segments.
4. Generate synthetic observations only from the session-independent training partition.
5. Select the classification threshold using validation data only.
6. Evaluate the test partition once after locking the complete protocol.
7. Keep training disabled until all split and leakage checks pass.

The synthetic observations remain transformation-derived children of real training samples; they do not represent additional independent radar sessions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42

TRAIN_FRACTION = 0.80
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.10

EXPECTED_SESSION_COUNT = 130
EXPECTED_SEGMENT_COUNT = 75868
EXPECTED_EDGE_SEGMENT_COUNT = 67

# Safe default. Do not change it yet.
RUN_SESSION_TRAINING = False

assert np.isclose(
    TRAIN_FRACTION
    + VALIDATION_FRACTION
    + TEST_FRACTION,
    1.0
)

print(
    "Session-split seed:",
    RANDOM_SEED
)

print(
    "Target split fractions:",
    {
        "training": TRAIN_FRACTION,
        "validation": VALIDATION_FRACTION,
        "test": TEST_FRACTION
    }
)

print(
    "Run session-independent training:",
    RUN_SESSION_TRAINING
)

## 1. Raw Session Inventory and Eligibility

The raw session structure is reconstructed directly from the SAAB SIRS file. Only bird and drone sessions are eligible for the binary classification experiment; corner-reflector and human sessions are documented and excluded. Edge-flagged segments are excluded before partition totals are calculated.

In [ ]:
RAW_DATA_CANDIDATES = [
    Path(
        "../data/raw/"
        "data_SAAB_SIRS_77GHz_FMCW.npy"
    ),
    Path(
        "../data/"
        "data_SAAB_SIRS_77GHz_FMCW.npy"
    ),
    Path(
        "../data/raw/SAAB_SIRS/"
        "data_SAAB_SIRS_77GHz_FMCW.npy"
    )
]

existing_raw_paths = [
    path
    for path in RAW_DATA_CANDIDATES
    if path.exists()
]

if len(existing_raw_paths) == 0:
    raise FileNotFoundError(
        "The raw SAAB SIRS dataset was not found. "
        "Checked:\n"
        + "\n".join(
            str(path.resolve())
            for path in RAW_DATA_CANDIDATES
        )
    )

if len(existing_raw_paths) > 1:
    print(
        "Multiple candidate files were found. "
        "Using the first one."
    )

RAW_DATA_PATH = existing_raw_paths[0]

raw_dataset = np.load(
    RAW_DATA_PATH,
    allow_pickle=True
)

assert raw_dataset.ndim == 2
assert raw_dataset.shape[1] == 6

print(
    "Raw dataset path:",
    RAW_DATA_PATH.resolve()
)

print(
    "Raw dataset shape:",
    raw_dataset.shape
)

print(
    "Raw dataset dtype:",
    raw_dataset.dtype
)

In [ ]:
DRONE_LABELS = {
    "D1",
    "D2",
    "D3",
    "D4",
    "D5",
    "D6"
}

BIRD_LABELS = {
    "black-headed gull",
    "heron",
    "pigeon",
    "raven",
    "seagull",
    "seagull and black-headed gull"
}

EXCLUDED_LABELS = {
    "CR",
    "human_walk",
    "human_run"
}


def normalize_original_label(value):
    flattened_value = np.asarray(
        value,
        dtype=object
    ).reshape(-1)

    if len(flattened_value) != 1:
        raise ValueError(
            "Expected exactly one label, "
            f"received: {value}"
        )

    scalar_value = flattened_value[0]

    if isinstance(
        scalar_value,
        bytes
    ):
        scalar_value = (
            scalar_value.decode(
                "utf-8"
            )
        )

    return str(
        scalar_value
    ).strip()


def assign_target_group(
    original_label
):
    if original_label in DRONE_LABELS:
        return "drone"

    if original_label in BIRD_LABELS:
        return "bird"

    if original_label in EXCLUDED_LABELS:
        return "excluded"

    raise ValueError(
        "Unknown original label: "
        + original_label
    )


session_records = []

for session_index, row in enumerate(
    raw_dataset
):
    original_label = (
        normalize_original_label(
            row[0]
        )
    )

    segments = np.asarray(
        row[1]
    )

    target_ranges = np.asarray(
        row[2]
    ).reshape(-1)

    timestamps = np.asarray(
        row[3]
    ).reshape(-1)

    official_splits = np.asarray(
        row[4]
    ).reshape(-1)

    edge_flags = np.asarray(
        row[5]
    ).reshape(-1)

    if segments.ndim != 2:
        raise ValueError(
            f"Session {session_index} has "
            "an invalid segment shape: "
            f"{segments.shape}"
        )

    if segments.shape[0] != 1280:
        raise ValueError(
            f"Session {session_index} does "
            "not contain 1,280 values "
            "per segment: "
            f"{segments.shape}"
        )

    number_of_segments = int(
        segments.shape[1]
    )

    for array_name, array in {
        "target_ranges": target_ranges,
        "timestamps": timestamps,
        "official_splits":
            official_splits,
        "edge_flags": edge_flags
    }.items():
        if len(array) != number_of_segments:
            raise ValueError(
                f"Session {session_index}: "
                f"{array_name} has length "
                f"{len(array)}, expected "
                f"{number_of_segments}."
            )

    edge_mask = (
        edge_flags.astype(bool)
    )

    usable_mask = ~edge_mask

    usable_official_splits = (
        official_splits[
            usable_mask
        ]
    )

    session_records.append({
        "session_index":
            session_index,

        "original_label":
            original_label,

        "target_group":
            assign_target_group(
                original_label
            ),

        "total_segments":
            number_of_segments,

        "edge_segments":
            int(edge_mask.sum()),

        "usable_segments":
            int(usable_mask.sum()),

        "official_split_1_segments":
            int(
                np.sum(
                    usable_official_splits
                    == 1
                )
            ),

        "official_split_2_segments":
            int(
                np.sum(
                    usable_official_splits
                    == 2
                )
            ),

        "official_split_3_segments":
            int(
                np.sum(
                    usable_official_splits
                    == 3
                )
            ),

        "minimum_range":
            float(
                np.nanmin(
                    target_ranges[
                        usable_mask
                    ]
                )
            ),

        "maximum_range":
            float(
                np.nanmax(
                    target_ranges[
                        usable_mask
                    ]
                )
            )
    })

session_summary_df = pd.DataFrame(
    session_records
)

eligible_session_summary_df = (
    session_summary_df[
        session_summary_df[
            "target_group"
        ].isin([
            "bird",
            "drone"
        ])
    ]
    .copy()
    .reset_index(drop=True)
)

excluded_session_summary_df = (
    session_summary_df[
        session_summary_df[
            "target_group"
        ] == "excluded"
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    session_summary_df.head()
)

In [ ]:
total_sessions = len(
    session_summary_df
)

total_segments = int(
    session_summary_df[
        "total_segments"
    ].sum()
)

total_edge_segments = int(
    session_summary_df[
        "edge_segments"
    ].sum()
)

total_usable_segments = int(
    session_summary_df[
        "usable_segments"
    ].sum()
)

assert total_sessions == 130
assert total_segments == 75868
assert total_edge_segments == 67
assert total_usable_segments == 75801

assert (
    session_summary_df[
        "session_index"
    ].is_unique
)

raw_group_summary_df = (
    session_summary_df
    .groupby(
        "target_group",
        observed=True
    )
    .agg(
        sessions=(
            "session_index",
            "count"
        ),
        total_segments=(
            "total_segments",
            "sum"
        ),
        edge_segments=(
            "edge_segments",
            "sum"
        ),
        usable_segments=(
            "usable_segments",
            "sum"
        )
    )
    .reset_index()
)

binary_group_summary_df = (
    eligible_session_summary_df
    .groupby(
        "target_group",
        observed=True
    )
    .agg(
        sessions=(
            "session_index",
            "count"
        ),
        usable_segments=(
            "usable_segments",
            "sum"
        )
    )
    .reset_index()
)

binary_subtype_summary_df = (
    eligible_session_summary_df
    .groupby(
        [
            "target_group",
            "original_label"
        ],
        observed=True
    )
    .agg(
        sessions=(
            "session_index",
            "count"
        ),
        usable_segments=(
            "usable_segments",
            "sum"
        ),
        minimum_session_size=(
            "usable_segments",
            "min"
        ),
        maximum_session_size=(
            "usable_segments",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        [
            "target_group",
            "usable_segments"
        ],
        ascending=[
            True,
            False
        ]
    )
)

official_binary_split_totals = {
    "training": int(
        eligible_session_summary_df[
            "official_split_1_segments"
        ].sum()
    ),
    "validation": int(
        eligible_session_summary_df[
            "official_split_2_segments"
        ].sum()
    ),
    "test": int(
        eligible_session_summary_df[
            "official_split_3_segments"
        ].sum()
    )
}

assert len(
    eligible_session_summary_df
) == 100

assert len(
    excluded_session_summary_df
) == 30

assert (
    eligible_session_summary_df[
        "usable_segments"
    ].sum()
    == 66493
)

assert official_binary_split_totals == {
    "training": 52509,
    "validation": 6988,
    "test": 6996
}

print(
    "Raw sessions:",
    total_sessions
)

print(
    "Eligible bird/drone sessions:",
    len(
        eligible_session_summary_df
    )
)

print(
    "Excluded CR/human sessions:",
    len(
        excluded_session_summary_df
    )
)

print(
    "Eligible bird/drone segments:",
    int(
        eligible_session_summary_df[
            "usable_segments"
        ].sum()
    )
)

print(
    "Official binary split totals:",
    official_binary_split_totals
)

display(
    raw_group_summary_df
)

display(
    binary_group_summary_df
)

display(
    binary_subtype_summary_df
)

print(
    "The corrected binary-session "
    "inventory passed all checks."
)

## 2. Deterministic Session-Independent Split

A deterministic randomized search is used to assign complete recording sessions to training, validation, and test partitions.

The optimization targets:

- an 80%/10%/10% session allocation;
- approximately 80%/10%/10% of usable segments overall;
- similar proportions within the bird and drone groups;
- representation of every recurrent bird subtype in both validation and test;
- four distinct drone models in each evaluation partition and all six models across validation and test;
- complete subtype coverage in training.

Pigeon and raven each contain only one session and are therefore retained in training. No signal values, model predictions, or test performance are used when constructing the split.

In [ ]:
SPLIT_SEARCH_ITERATIONS = 1000000

TARGET_SESSION_COUNTS = {
    "training": {
        "bird": 44,
        "drone": 36
    },
    "validation": {
        "bird": 6,
        "drone": 4
    },
    "test": {
        "bird": 6,
        "drone": 4
    }
}

split_source_df = (
    eligible_session_summary_df[
        [
            "session_index",
            "original_label",
            "target_group",
            "usable_segments"
        ]
    ]
    .copy()
    .sort_values(
        "session_index"
    )
    .reset_index(drop=True)
)

session_indices = (
    split_source_df[
        "session_index"
    ].to_numpy(
        dtype=np.int64
    )
)

original_labels = (
    split_source_df[
        "original_label"
    ].to_numpy()
)

target_groups = (
    split_source_df[
        "target_group"
    ].to_numpy()
)

usable_segment_counts = (
    split_source_df[
        "usable_segments"
    ].to_numpy(
        dtype=np.int64
    )
)

bird_positions = np.flatnonzero(
    target_groups == "bird"
)

drone_positions = np.flatnonzero(
    target_groups == "drone"
)

label_session_counts = (
    split_source_df[
        "original_label"
    ]
    .value_counts()
)

singleton_labels = set(
    label_session_counts[
        label_session_counts == 1
    ].index
)

forced_training_positions = (
    np.flatnonzero(
        np.isin(
            original_labels,
            list(singleton_labels)
        )
    )
)

available_bird_positions = (
    np.setdiff1d(
        bird_positions,
        forced_training_positions
    )
)

all_original_labels = set(
    original_labels
)

required_evaluation_bird_labels = {
    label
    for label in BIRD_LABELS
    if label_session_counts[
        label
    ] >= 2
}

search_rng = np.random.default_rng(
    RANDOM_SEED
)

best_score = np.inf
best_assignment = None
valid_candidate_count = 0

for search_iteration in range(
    SPLIT_SEARCH_ITERATIONS
):
    validation_bird_positions = (
        search_rng.choice(
            available_bird_positions,
            size=6,
            replace=False
        )
    )

    remaining_bird_positions = (
        np.setdiff1d(
            available_bird_positions,
            validation_bird_positions
        )
    )

    test_bird_positions = (
        search_rng.choice(
            remaining_bird_positions,
            size=6,
            replace=False
        )
    )

    validation_drone_positions = (
        search_rng.choice(
            drone_positions,
            size=4,
            replace=False
        )
    )

    remaining_drone_positions = (
        np.setdiff1d(
            drone_positions,
            validation_drone_positions
        )
    )

    test_drone_positions = (
        search_rng.choice(
            remaining_drone_positions,
            size=4,
            replace=False
        )
    )

    validation_positions = (
        np.concatenate([
            validation_bird_positions,
            validation_drone_positions
        ])
    )

    test_positions = (
        np.concatenate([
            test_bird_positions,
            test_drone_positions
        ])
    )

    training_positions = (
        np.setdiff1d(
            np.arange(
                len(split_source_df)
            ),
            np.concatenate([
                validation_positions,
                test_positions
            ])
        )
    )

    training_labels = set(
        original_labels[
            training_positions
        ]
    )

    validation_bird_labels = set(
        original_labels[
            validation_bird_positions
        ]
    )

    validation_drone_labels = set(
        original_labels[
            validation_drone_positions
        ]
    )

    test_bird_labels = set(
        original_labels[
            test_bird_positions
        ]
    )

    test_drone_labels = set(
        original_labels[
            test_drone_positions
        ]
    )

    # Every original subtype must remain
    # represented in the training partition.
    if training_labels != all_original_labels:
        continue

    # Pigeon and raven contain only one
    # session and remain in training.
    # Every recurrent bird subtype must
    # occur in validation.
    if not (
        required_evaluation_bird_labels
        .issubset(
            validation_bird_labels
        )
    ):
        continue

    # Every recurrent bird subtype must
    # also occur in test.
    if not (
        required_evaluation_bird_labels
        .issubset(
            test_bird_labels
        )
    ):
        continue

    # Validation contains four drone
    # sessions. Require four different
    # drone models.
    if len(
        validation_drone_labels
    ) != 4:
        continue

    # Test also contains four different
    # drone models.
    if len(
        test_drone_labels
    ) != 4:
        continue

    # Across validation and test together,
    # all six drone models must occur.
    evaluation_drone_labels = (
        validation_drone_labels
        | test_drone_labels
    )

    if not DRONE_LABELS.issubset(
        evaluation_drone_labels
    ):
        continue

    valid_candidate_count += 1

    candidate_assignment = np.full(
        len(split_source_df),
        "training",
        dtype=object
    )

    candidate_assignment[
        validation_positions
    ] = "validation"

    candidate_assignment[
        test_positions
    ] = "test"

    candidate_score = 0.0

    for population_mask in [
        np.ones(
            len(split_source_df),
            dtype=bool
        ),
        target_groups == "bird",
        target_groups == "drone"
    ]:
        population_total = float(
            usable_segment_counts[
                population_mask
            ].sum()
        )

        for (
            partition_name,
            target_fraction
        ) in {
            "training": 0.80,
            "validation": 0.10,
            "test": 0.10
        }.items():
            observed_segments = float(
                usable_segment_counts[
                    population_mask
                    & (
                        candidate_assignment
                        == partition_name
                    )
                ].sum()
            )

            observed_fraction = (
                observed_segments
                / population_total
            )

            normalized_error = (
                observed_fraction
                - target_fraction
            ) / target_fraction

            candidate_score += (
                normalized_error ** 2
            )

    if candidate_score < best_score:
        best_score = candidate_score

        best_assignment = (
            candidate_assignment.copy()
        )

assert best_assignment is not None

assert valid_candidate_count > 0

session_assignment_df = (
    split_source_df.copy()
)

session_assignment_df[
    "session_partition"
] = best_assignment

session_assignment_df[
    "split_seed"
] = RANDOM_SEED

print(
    "Valid candidate splits evaluated:",
    valid_candidate_count
)

print(
    "Selected split score:",
    best_score
)

In [ ]:
partition_order = [
    "training",
    "validation",
    "test"
]

assert len(
    session_assignment_df
) == 100

assert (
    session_assignment_df[
        "session_index"
    ].is_unique
)

assert set(
    session_assignment_df[
        "session_partition"
    ]
) == set(partition_order)

training_session_ids = set(
    session_assignment_df.loc[
        session_assignment_df[
            "session_partition"
        ] == "training",
        "session_index"
    ]
)

validation_session_ids = set(
    session_assignment_df.loc[
        session_assignment_df[
            "session_partition"
        ] == "validation",
        "session_index"
    ]
)

test_session_ids = set(
    session_assignment_df.loc[
        session_assignment_df[
            "session_partition"
        ] == "test",
        "session_index"
    ]
)

assert training_session_ids.isdisjoint(
    validation_session_ids
)

assert training_session_ids.isdisjoint(
    test_session_ids
)

assert validation_session_ids.isdisjoint(
    test_session_ids
)

assert (
    training_session_ids
    | validation_session_ids
    | test_session_ids
) == set(
    session_assignment_df[
        "session_index"
    ]
)

split_group_summary_df = (
    session_assignment_df
    .groupby(
        [
            "session_partition",
            "target_group"
        ],
        observed=True
    )
    .agg(
        sessions=(
            "session_index",
            "count"
        ),
        usable_segments=(
            "usable_segments",
            "sum"
        )
    )
    .reset_index()
)

split_group_summary_df[
    "within_group_fraction"
] = (
    split_group_summary_df[
        "usable_segments"
    ]
    / split_group_summary_df
    .groupby(
        "target_group"
    )["usable_segments"]
    .transform("sum")
)

split_overall_summary_df = (
    session_assignment_df
    .groupby(
        "session_partition",
        observed=True
    )
    .agg(
        sessions=(
            "session_index",
            "count"
        ),
        usable_segments=(
            "usable_segments",
            "sum"
        )
    )
    .reindex(
        partition_order
    )
    .reset_index()
)

split_overall_summary_df[
    "session_fraction"
] = (
    split_overall_summary_df[
        "sessions"
    ]
    / split_overall_summary_df[
        "sessions"
    ].sum()
)

split_overall_summary_df[
    "segment_fraction"
] = (
    split_overall_summary_df[
        "usable_segments"
    ]
    / split_overall_summary_df[
        "usable_segments"
    ].sum()
)

split_subtype_summary_df = (
    session_assignment_df
    .pivot_table(
        index=[
            "target_group",
            "original_label"
        ],
        columns="session_partition",
        values="session_index",
        aggfunc="count",
        fill_value=0
    )
    .reindex(
        columns=partition_order,
        fill_value=0
    )
    .reset_index()
)

expected_session_counts = {
    (
        partition_name,
        target_group
    ): expected_count
    for partition_name, group_counts
    in TARGET_SESSION_COUNTS.items()
    for target_group, expected_count
    in group_counts.items()
}

observed_session_counts = {
    (
        row[
            "session_partition"
        ],
        row["target_group"]
    ): int(row["sessions"])
    for _, row
    in split_group_summary_df.iterrows()
}

assert (
    observed_session_counts
    == expected_session_counts
)

assert set(
    session_assignment_df.loc[
        session_assignment_df[
            "session_partition"
        ] == "training",
        "original_label"
    ]
) == all_original_labels

print(
    "No session overlap was detected."
)

display(
    split_overall_summary_df.style.format({
        "session_fraction": "{:.2%}",
        "segment_fraction": "{:.2%}"
    })
)

display(
    split_group_summary_df.style.format({
        "within_group_fraction":
            "{:.2%}"
    })
)

display(
    split_subtype_summary_df
)

In [ ]:
SESSION_SPLIT_DIR = Path(
    "../data/processed/"
    "session_independent_split"
)

SESSION_SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SESSION_ASSIGNMENT_PATH = (
    SESSION_SPLIT_DIR
    / (
        "session_assignments_"
        f"seed_{RANDOM_SEED}_"
        "subtype_balanced.csv"
    )
)

assignment_columns = [
    "session_index",
    "original_label",
    "target_group",
    "usable_segments",
    "session_partition",
    "split_seed"
]

if SESSION_ASSIGNMENT_PATH.exists():
    saved_session_assignment_df = (
        pd.read_csv(
            SESSION_ASSIGNMENT_PATH
        )
    )

    pd.testing.assert_frame_equal(
        saved_session_assignment_df[
            assignment_columns
        ].reset_index(drop=True),
        session_assignment_df[
            assignment_columns
        ].reset_index(drop=True),
        check_dtype=False
    )

    print(
        "Existing subtype-balanced "
        "session assignment was "
        "reloaded and verified."
    )

else:
    session_assignment_df.to_csv(
        SESSION_ASSIGNMENT_PATH,
        index=False
    )

    print(
        "Subtype-balanced session "
        "assignment saved:",
        SESSION_ASSIGNMENT_PATH.resolve()
    )

print(
    "Locked split score:",
    best_score
)

print(
    "The session-independent split "
    "passed all leakage, balance, "
    "and subtype-coverage checks."
)

## 3. Session-Independent Tensor Generation

The locked session assignment is now converted into model-ready range–Doppler tensors.

The transformation is copied exactly from Notebook 04:

1. reshape every segment to `(5, 256)`;
2. retain azimuth samples 54–203;
3. apply a Hann window;
4. compute and centre the slow-time FFT;
5. express magnitude relative to the patch maximum;
6. clip the dynamic range to `[-60, 0]` dB;
7. normalize the result to `[0, 1]`;
8. store the output as `float32` with shape `(5, 150)`.

All usable segments from a recording session are assigned to the same partition, independently of their original official split indicators.

In [ ]:
import gc

AZIMUTH_START = 54
AZIMUTH_END = 204

MIN_DB = -60.0
MAX_DB = 0.0

LABEL_ENCODING = {
    "bird": 0,
    "drone": 1
}

OFFICIAL_PROCESSED_DATA_DIR = Path(
    "../data/processed/official_split"
)

SESSION_PROCESSED_DATA_DIR = (
    SESSION_SPLIT_DIR
)

required_official_files = [
    OFFICIAL_PROCESSED_DATA_DIR
    / "X_train.npy",

    OFFICIAL_PROCESSED_DATA_DIR
    / "metadata_train.csv"
]

missing_official_files = [
    path
    for path in required_official_files
    if not path.exists()
]

if missing_official_files:
    raise FileNotFoundError(
        "Missing official preprocessing "
        "reference files:\n"
        + "\n".join(
            str(path.resolve())
            for path in missing_official_files
        )
    )


def transform_segment_batch(
    segment_batch
):
    """
    Convert (N, 5, 256) complex segments
    into normalized float32 tensors with
    shape (N, 5, 150).
    """
    segment_batch = np.asarray(
        segment_batch
    )

    if (
        segment_batch.ndim != 3
        or segment_batch.shape[1:] != (
            5,
            256
        )
    ):
        raise ValueError(
            "Expected shape (N, 5, 256), "
            f"received {segment_batch.shape}"
        )

    central_samples = segment_batch[
        :,
        :,
        AZIMUTH_START:AZIMUTH_END
    ]

    window = np.hanning(
        central_samples.shape[2]
    ).astype(np.float32)

    windowed = (
        central_samples
        * window[
            np.newaxis,
            np.newaxis,
            :
        ]
    )

    doppler_complex = np.fft.fftshift(
        np.fft.fft(
            windowed,
            axis=2
        ),
        axes=2
    )

    magnitude_db = (
        20
        * np.log10(
            np.abs(
                doppler_complex
            )
            + 1e-12
        )
    )

    patch_maximum = np.max(
        magnitude_db,
        axis=(
            1,
            2
        ),
        keepdims=True
    )

    relative_db = (
        magnitude_db
        - patch_maximum
    )

    clipped_db = np.clip(
        relative_db,
        MIN_DB,
        MAX_DB
    )

    normalized = (
        (clipped_db - MIN_DB)
        / (MAX_DB - MIN_DB)
    )

    return normalized.astype(
        np.float32
    )


print(
    "Session-independent processing "
    "utilities are ready."
)

In [ ]:
official_X_train = np.load(
    OFFICIAL_PROCESSED_DATA_DIR
    / "X_train.npy",
    mmap_mode="r"
)

official_metadata_train = (
    pd.read_csv(
        OFFICIAL_PROCESSED_DATA_DIR
        / "metadata_train.csv"
    )
)

reference_metadata = (
    official_metadata_train.iloc[0]
)

reference_array_index = int(
    reference_metadata[
        "array_index"
    ]
)

reference_session_index = int(
    reference_metadata[
        "session_id"
    ]
)

reference_segment_index = int(
    reference_metadata[
        "segment_id"
    ]
)

reference_raw_segments = np.asarray(
    raw_dataset[
        reference_session_index,
        1
    ]
)

reference_segment_batch = (
    reference_raw_segments[
        :,
        [
            reference_segment_index
        ]
    ]
    .T
    .reshape(
        -1,
        5,
        256
    )
)

reprocessed_reference = (
    transform_segment_batch(
        reference_segment_batch
    )[0]
)

saved_reference = np.asarray(
    official_X_train[
        reference_array_index
    ]
)

maximum_reference_difference = float(
    np.max(
        np.abs(
            reprocessed_reference
            - saved_reference
        )
    )
)

assert np.allclose(
    reprocessed_reference,
    saved_reference,
    rtol=0.0,
    atol=1e-7
)

assert (
    reprocessed_reference.dtype
    == np.float32
)

assert (
    reprocessed_reference.shape
    == (
        5,
        150
    )
)

print(
    "Reference session:",
    reference_session_index
)

print(
    "Reference segment:",
    reference_segment_index
)

print(
    "Maximum difference from "
    "Notebook 04:",
    maximum_reference_difference
)

print(
    "The preprocessing implementation "
    "exactly matches Notebook 04."
)

In [ ]:
PARTITION_FILE_NAMES = {
    "training": "train",
    "validation": "validation",
    "test": "test"
}

expected_partition_counts = {
    partition_name: int(
        session_assignment_df.loc[
            session_assignment_df[
                "session_partition"
            ] == partition_name,
            "usable_segments"
        ].sum()
    )
    for partition_name
    in PARTITION_FILE_NAMES
}

session_to_partition = dict(
    zip(
        session_assignment_df[
            "session_index"
        ].astype(int),
        session_assignment_df[
            "session_partition"
        ]
    )
)


def get_partition_output_paths(
    partition_name
):
    file_name = (
        PARTITION_FILE_NAMES[
            partition_name
        ]
    )

    return {
        "features":
            SESSION_PROCESSED_DATA_DIR
            / f"X_{file_name}.npy",

        "labels":
            SESSION_PROCESSED_DATA_DIR
            / f"y_{file_name}.npy",

        "metadata":
            SESSION_PROCESSED_DATA_DIR
            / f"metadata_{file_name}.csv"
    }


def process_session_partition(
    partition_name
):
    expected_count = (
        expected_partition_counts[
            partition_name
        ]
    )

    output_paths = (
        get_partition_output_paths(
            partition_name
        )
    )

    artifact_exists = {
        artifact_name: path.exists()
        for artifact_name, path
        in output_paths.items()
    }

    if any(
        artifact_exists.values()
    ) and not all(
        artifact_exists.values()
    ):
        raise RuntimeError(
            "Incomplete existing artifacts "
            f"for {partition_name}: "
            + str(artifact_exists)
        )

    if all(
        artifact_exists.values()
    ):
        X_saved = np.load(
            output_paths["features"],
            mmap_mode="r"
        )

        y_saved = np.load(
            output_paths["labels"],
            mmap_mode="r"
        )

        metadata_saved = pd.read_csv(
            output_paths["metadata"]
        )

        assert X_saved.shape == (
            expected_count,
            5,
            150
        )

        assert y_saved.shape == (
            expected_count,
        )

        assert len(
            metadata_saved
        ) == expected_count

        expected_sessions = set(
            session_assignment_df.loc[
                session_assignment_df[
                    "session_partition"
                ] == partition_name,
                "session_index"
            ].astype(int)
        )

        saved_sessions = set(
            metadata_saved[
                "session_index"
            ].astype(int)
        )

        assert (
            saved_sessions
            == expected_sessions
        )

        print(
            partition_name,
            "artifacts already exist and "
            "were verified."
        )

        return (
            X_saved,
            y_saved,
            metadata_saved
        )

    X = np.empty(
        (
            expected_count,
            5,
            150
        ),
        dtype=np.float32
    )

    y = np.empty(
        expected_count,
        dtype=np.uint8
    )

    metadata_records = []
    output_position = 0

    for session_index, row in enumerate(
        raw_dataset
    ):
        if (
            session_to_partition.get(
                session_index
            )
            != partition_name
        ):
            continue

        original_label = (
            normalize_original_label(
                row[0]
            )
        )

        target_group = (
            assign_target_group(
                original_label
            )
        )

        if target_group not in {
            "bird",
            "drone"
        }:
            raise RuntimeError(
                "An excluded session entered "
                f"{partition_name}: "
                f"{session_index}"
            )

        segments = np.asarray(
            row[1]
        )

        ranges = np.asarray(
            row[2]
        ).reshape(-1)

        timestamps = np.asarray(
            row[3]
        ).reshape(-1)

        official_splits = np.asarray(
            row[4]
        ).reshape(-1)

        edge_flags = np.asarray(
            row[5]
        ).reshape(-1)

        selected_indices = np.flatnonzero(
            edge_flags == 0
        )

        if selected_indices.size == 0:
            continue

        segment_batch = (
            segments[
                :,
                selected_indices
            ]
            .T
            .reshape(
                -1,
                5,
                256
            )
        )

        transformed_batch = (
            transform_segment_batch(
                segment_batch
            )
        )

        batch_size = int(
            len(selected_indices)
        )

        batch_end = (
            output_position
            + batch_size
        )

        if batch_end > expected_count:
            raise RuntimeError(
                f"{partition_name}: processed "
                "more samples than expected."
            )

        X[
            output_position:batch_end
        ] = transformed_batch

        y[
            output_position:batch_end
        ] = LABEL_ENCODING[
            target_group
        ]

        for (
            local_position,
            segment_index
        ) in enumerate(
            selected_indices
        ):
            metadata_records.append({
                "array_index":
                    (
                        output_position
                        + local_position
                    ),

                "session_index":
                    session_index,

                "segment_index":
                    int(
                        segment_index
                    ),

                "original_label":
                    original_label,

                "target_group":
                    target_group,

                "encoded_label":
                    LABEL_ENCODING[
                        target_group
                    ],

                "range_m":
                    float(
                        ranges[
                            segment_index
                        ]
                    ),

                "time_s":
                    float(
                        timestamps[
                            segment_index
                        ]
                    ),

                "original_official_split":
                    int(
                        official_splits[
                            segment_index
                        ]
                    ),

                "session_partition":
                    partition_name,

                "edge_flag":
                    int(
                        edge_flags[
                            segment_index
                        ]
                    )
            })

        output_position = batch_end

    if output_position != expected_count:
        raise RuntimeError(
            f"{partition_name}: expected "
            f"{expected_count} samples, "
            f"processed {output_position}."
        )

    metadata_df = pd.DataFrame(
        metadata_records
    )

    assert len(
        metadata_df
    ) == expected_count

    assert np.isfinite(
        X
    ).all()

    assert X.min() >= 0.0
    assert X.max() <= 1.0

    assert np.array_equal(
        y,
        metadata_df[
            "encoded_label"
        ].to_numpy(
            dtype=np.uint8
        )
    )

    np.save(
        output_paths["features"],
        X
    )

    np.save(
        output_paths["labels"],
        y
    )

    metadata_df.to_csv(
        output_paths["metadata"],
        index=False
    )

    print(partition_name + ":")

    print(
        "  X shape:",
        X.shape
    )

    print(
        "  y shape:",
        y.shape
    )

    print(
        "  Sessions:",
        metadata_df[
            "session_index"
        ].nunique()
    )

    print(
        "  Class counts [bird, drone]:",
        np.bincount(
            y,
            minlength=2
        )
    )

    print(
        "  Value range:",
        float(X.min()),
        "to",
        float(X.max())
    )

    return (
        X,
        y,
        metadata_df
    )

In [ ]:
partition_processing_records = []

for partition_name in [
    "training",
    "validation",
    "test"
]:
    (
        X_partition,
        y_partition,
        metadata_partition
    ) = process_session_partition(
        partition_name
    )

    partition_processing_records.append({
        "partition":
            partition_name,

        "samples":
            int(
                len(y_partition)
            ),

        "sessions":
            int(
                metadata_partition[
                    "session_index"
                ].nunique()
            ),

        "birds":
            int(
                np.sum(
                    np.asarray(
                        y_partition
                    ) == 0
                )
            ),

        "drones":
            int(
                np.sum(
                    np.asarray(
                        y_partition
                    ) == 1
                )
            ),

        "minimum_value":
            float(
                np.min(
                    X_partition
                )
            ),

        "maximum_value":
            float(
                np.max(
                    X_partition
                )
            )
    })

    del X_partition
    del y_partition
    del metadata_partition

    gc.collect()

partition_processing_df = pd.DataFrame(
    partition_processing_records
)

display(
    partition_processing_df
)

assert (
    partition_processing_df[
        "samples"
    ].sum()
    == 66493
)

assert (
    partition_processing_df[
        "sessions"
    ].sum()
    == 100
)

print(
    "All session-independent tensors "
    "were generated or safely verified."
)

## 4. Balanced 10% Real Training Subset

The session-independent training partition contains 6,236 birds and 46,775 drones.

As in the official-split experiments, the limited-data subset is defined relative to the largest balanced training set. Ten percent of the minority-class capacity gives 624 samples per class after rounding upward.

The resulting real training subset therefore contains:

- 624 bird observations;
- 624 drone observations;
- 1,248 observations in total.

Selection is deterministic with seed 42 and uses only the session-independent training partition. Validation and test observations cannot become augmentation parents.

In [ ]:
from math import ceil

SUBSET_SELECTION_SEED = 42
REAL_SUBSET_FRACTION = 0.10

SESSION_LIMITED_DATA_DIR = (
    SESSION_SPLIT_DIR
    / "limited_subsets"
)

SESSION_LIMITED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REAL_SUBSET_INDEX_PATH = (
    SESSION_LIMITED_DATA_DIR
    / "indices_10_percent_seed_42.npy"
)

REAL_SUBSET_METADATA_PATH = (
    SESSION_LIMITED_DATA_DIR
    / "metadata_10_percent_seed_42.csv"
)

X_session_train = np.load(
    SESSION_PROCESSED_DATA_DIR
    / "X_train.npy",
    mmap_mode="r"
)

y_session_train = np.load(
    SESSION_PROCESSED_DATA_DIR
    / "y_train.npy",
    mmap_mode="r"
)

metadata_session_train = pd.read_csv(
    SESSION_PROCESSED_DATA_DIR
    / "metadata_train.csv"
)

assert X_session_train.shape == (
    53011,
    5,
    150
)

assert y_session_train.shape == (
    53011,
)

assert len(
    metadata_session_train
) == 53011

training_class_counts = np.bincount(
    y_session_train,
    minlength=2
)

balanced_training_capacity = int(
    training_class_counts.min()
)

real_samples_per_class = int(
    ceil(
        balanced_training_capacity
        * REAL_SUBSET_FRACTION
    )
)

assert (
    balanced_training_capacity
    == 6236
)

assert (
    real_samples_per_class
    == 624
)

subset_rng = np.random.default_rng(
    SUBSET_SELECTION_SEED
)

bird_training_indices = np.flatnonzero(
    y_session_train == 0
)

drone_training_indices = np.flatnonzero(
    y_session_train == 1
)

selected_bird_indices = (
    subset_rng.permutation(
        bird_training_indices
    )[
        :real_samples_per_class
    ]
)

selected_drone_indices = (
    subset_rng.permutation(
        drone_training_indices
    )[
        :real_samples_per_class
    ]
)

generated_real_subset_indices = (
    np.sort(
        np.concatenate([
            selected_bird_indices,
            selected_drone_indices
        ])
    )
)

assert len(
    generated_real_subset_indices
) == 1248

assert len(
    np.unique(
        generated_real_subset_indices
    )
) == 1248

generated_subset_metadata = (
    metadata_session_train
    .iloc[
        generated_real_subset_indices
    ]
    .copy()
    .reset_index(drop=True)
)

generated_subset_metadata.insert(
    0,
    "source_training_array_index",
    generated_real_subset_indices
)

existing_subset_artifacts = {
    "indices":
        REAL_SUBSET_INDEX_PATH.exists(),

    "metadata":
        REAL_SUBSET_METADATA_PATH.exists()
}

if any(
    existing_subset_artifacts.values()
) and not all(
    existing_subset_artifacts.values()
):
    raise RuntimeError(
        "Incomplete existing limited-subset "
        "artifacts: "
        + str(
            existing_subset_artifacts
        )
    )

if all(
    existing_subset_artifacts.values()
):
    saved_real_subset_indices = np.load(
        REAL_SUBSET_INDEX_PATH
    )

    saved_subset_metadata = pd.read_csv(
        REAL_SUBSET_METADATA_PATH
    )

    assert np.array_equal(
        saved_real_subset_indices,
        generated_real_subset_indices
    )

    assert np.array_equal(
        saved_subset_metadata[
            "source_training_array_index"
        ].to_numpy(
            dtype=np.int64
        ),
        generated_real_subset_indices
    )

    real_subset_indices = (
        saved_real_subset_indices
    )

    real_subset_metadata = (
        saved_subset_metadata
    )

    print(
        "Existing 10% real subset was "
        "reloaded and verified."
    )

else:
    np.save(
        REAL_SUBSET_INDEX_PATH,
        generated_real_subset_indices
    )

    generated_subset_metadata.to_csv(
        REAL_SUBSET_METADATA_PATH,
        index=False
    )

    real_subset_indices = (
        generated_real_subset_indices
    )

    real_subset_metadata = (
        generated_subset_metadata
    )

    print(
        "Deterministic 10% real subset "
        "was saved."
    )

In [ ]:
X_real_source = np.asarray(
    X_session_train[
        real_subset_indices
    ],
    dtype=np.float32
)

y_real_source = np.asarray(
    y_session_train[
        real_subset_indices
    ],
    dtype=np.uint8
)

assert X_real_source.shape == (
    1248,
    5,
    150
)

assert y_real_source.shape == (
    1248,
)

assert np.array_equal(
    np.bincount(
        y_real_source,
        minlength=2
    ),
    [
        624,
        624
    ]
)

assert np.isfinite(
    X_real_source
).all()

assert X_real_source.min() >= 0.0
assert X_real_source.max() <= 1.0

assert (
    real_subset_metadata[
        "session_partition"
    ] == "training"
).all()

source_session_ids = set(
    real_subset_metadata[
        "session_index"
    ].astype(int)
)

assert source_session_ids.issubset(
    training_session_ids
)

assert source_session_ids.isdisjoint(
    validation_session_ids
)

assert source_session_ids.isdisjoint(
    test_session_ids
)

real_subset_summary_df = (
    real_subset_metadata
    .groupby(
        "target_group",
        observed=True
    )
    .agg(
        samples=(
            "source_training_array_index",
            "count"
        ),
        represented_sessions=(
            "session_index",
            "nunique"
        ),
        represented_subtypes=(
            "original_label",
            "nunique"
        )
    )
    .reset_index()
)

print(
    "Real source tensor:",
    X_real_source.shape
)

print(
    "Real source class counts "
    "[bird, drone]:",
    np.bincount(
        y_real_source,
        minlength=2
    )
)

print(
    "Total represented sessions:",
    len(source_session_ids)
)

display(
    real_subset_summary_df
)

print(
    "The 10% real source subset passed "
    "all training-only leakage checks."
)

## 5. Training-Only Synthetic Augmentation

The selected 1:1 augmentation policy from Notebook 10 is now applied to the session-independent experiment.

Exactly one synthetic child is generated for every real observation in the balanced 10% training subset. Therefore:

- real parents: 1,248;
- synthetic children: 1,248;
- augmented training observations: 2,496;
- birds per source: 624 real and 624 synthetic;
- drones per source: 624 real and 624 synthetic.

All parents originate exclusively from the 80 training sessions. Validation and test sessions cannot contribute augmentation parents.

In [ ]:
import hashlib
import json


def shift_feature_axis(
    sample,
    shift
):
    """
    Translate a (5, 150) tensor along
    its feature axis using nearest-edge
    filling and no circular wrapping.
    """
    if shift == 0:
        return sample.copy()

    shifted = np.empty_like(
        sample
    )

    if shift > 0:
        shifted[
            :,
            :shift
        ] = sample[
            :,
            :1
        ]

        shifted[
            :,
            shift:
        ] = sample[
            :,
            :-shift
        ]

    else:
        width = abs(
            shift
        )

        shifted[
            :,
            -width:
        ] = sample[
            :,
            -1:
        ]

        shifted[
            :,
            :-width
        ] = sample[
            :,
            width:
        ]

    return shifted


def augment_radar_sample(
    sample,
    generator
):
    """
    Generate one transformation-based
    synthetic child from a real parent.
    """
    synthetic_sample = np.asarray(
        sample,
        dtype=np.float32
    ).copy()

    parameters = {}

    feature_shift = int(
        generator.integers(
            -5,
            6
        )
    )

    synthetic_sample = (
        shift_feature_axis(
            synthetic_sample,
            feature_shift
        )
    )

    parameters[
        "feature_shift"
    ] = feature_shift

    amplitude_scale = float(
        generator.uniform(
            0.90,
            1.10
        )
    )

    synthetic_sample *= (
        amplitude_scale
    )

    parameters[
        "amplitude_scale"
    ] = amplitude_scale

    contrast_factor = float(
        generator.uniform(
            0.90,
            1.10
        )
    )

    sample_mean = float(
        synthetic_sample.mean()
    )

    synthetic_sample = (
        sample_mean
        + contrast_factor
        * (
            synthetic_sample
            - sample_mean
        )
    )

    parameters[
        "contrast_factor"
    ] = contrast_factor

    noise_standard_deviation = float(
        generator.uniform(
            0.002,
            0.015
        )
    )

    noise = generator.normal(
        loc=0.0,
        scale=(
            noise_standard_deviation
        ),
        size=synthetic_sample.shape
    ).astype(np.float32)

    synthetic_sample += noise

    parameters[
        "noise_standard_deviation"
    ] = (
        noise_standard_deviation
    )

    mask_applied = bool(
        generator.random()
        < 0.30
    )

    mask_start = -1
    mask_width = 0

    if mask_applied:
        mask_width = int(
            generator.integers(
                2,
                7
            )
        )

        mask_start = int(
            generator.integers(
                0,
                synthetic_sample.shape[1]
                - mask_width
                + 1
            )
        )

        local_fill_value = float(
            synthetic_sample.mean()
        )

        synthetic_sample[
            :,
            mask_start:
            mask_start
            + mask_width
        ] = local_fill_value

    parameters[
        "mask_applied"
    ] = mask_applied

    parameters[
        "mask_start"
    ] = mask_start

    parameters[
        "mask_width"
    ] = mask_width

    synthetic_sample = np.clip(
        synthetic_sample,
        0.0,
        1.0
    ).astype(np.float32)

    return (
        synthetic_sample,
        parameters
    )


def calculate_sha256(
    file_path
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

In [ ]:
SYNTHETIC_GENERATION_SEED = 42
SYNTHETIC_TO_REAL_RATIO = 1.0

generation_rng = (
    np.random.default_rng(
        SYNTHETIC_GENERATION_SEED
    )
)

synthetic_samples = []
synthetic_labels = []
synthetic_metadata_records = []

for local_parent_index, (
    real_sample,
    real_label
) in enumerate(
    zip(
        X_real_source,
        y_real_source
    )
):
    (
        synthetic_sample,
        transformation_parameters
    ) = augment_radar_sample(
        real_sample,
        generation_rng
    )

    parent_metadata = (
        real_subset_metadata.iloc[
            local_parent_index
        ]
    )

    synthetic_samples.append(
        synthetic_sample
    )

    synthetic_labels.append(
        real_label
    )

    synthetic_metadata_records.append({
        "synthetic_index":
            local_parent_index,

        "local_parent_index":
            local_parent_index,

        "parent_training_array_index":
            int(
                real_subset_indices[
                    local_parent_index
                ]
            ),

        "parent_session_index":
            int(
                parent_metadata[
                    "session_index"
                ]
            ),

        "parent_segment_index":
            int(
                parent_metadata[
                    "segment_index"
                ]
            ),

        "parent_original_label":
            parent_metadata[
                "original_label"
            ],

        "parent_target_group":
            parent_metadata[
                "target_group"
            ],

        "parent_label":
            int(real_label),

        "child_number":
            0,

        **transformation_parameters
    })

X_synthetic_session = np.stack(
    synthetic_samples
).astype(np.float32)

y_synthetic_session = np.asarray(
    synthetic_labels,
    dtype=np.uint8
)

metadata_synthetic_session = (
    pd.DataFrame(
        synthetic_metadata_records
    )
)

assert X_synthetic_session.shape == (
    1248,
    5,
    150
)

assert y_synthetic_session.shape == (
    1248,
)

assert len(
    metadata_synthetic_session
) == 1248

assert np.array_equal(
    np.bincount(
        y_synthetic_session,
        minlength=2
    ),
    [
        624,
        624
    ]
)

assert (
    X_synthetic_session.dtype
    == np.float32
)

assert (
    y_synthetic_session.dtype
    == np.uint8
)

assert np.isfinite(
    X_synthetic_session
).all()

assert (
    X_synthetic_session.min()
    >= 0.0
)

assert (
    X_synthetic_session.max()
    <= 1.0
)

assert np.array_equal(
    y_synthetic_session,
    metadata_synthetic_session[
        "parent_label"
    ].to_numpy(
        dtype=np.uint8
    )
)

synthetic_parent_sessions = set(
    metadata_synthetic_session[
        "parent_session_index"
    ].astype(int)
)

assert synthetic_parent_sessions.issubset(
    training_session_ids
)

assert synthetic_parent_sessions.isdisjoint(
    validation_session_ids
)

assert synthetic_parent_sessions.isdisjoint(
    test_session_ids
)

unchanged_children = np.all(
    X_synthetic_session
    == X_real_source,
    axis=(
        1,
        2
    )
)

assert not np.any(
    unchanged_children
)

print(
    "Synthetic tensor:",
    X_synthetic_session.shape
)

print(
    "Synthetic class counts "
    "[bird, drone]:",
    np.bincount(
        y_synthetic_session,
        minlength=2
    )
)

print(
    "Synthetic parent sessions:",
    len(
        synthetic_parent_sessions
    )
)

print(
    "Synthetic value range:",
    float(
        X_synthetic_session.min()
    ),
    "to",
    float(
        X_synthetic_session.max()
    )
)

display(
    metadata_synthetic_session.head()
)

print(
    "The session-independent synthetic "
    "dataset passed all structural and "
    "leakage checks."
)

In [ ]:
SESSION_SYNTHETIC_DATA_DIR = (
    SESSION_SPLIT_DIR
    / "synthetic_subsets"
    / (
        "10_percent_signal_augmentation_"
        "ratio_1_seed_42"
    )
)

SESSION_SYNTHETIC_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

session_synthetic_files = {
    "features":
        SESSION_SYNTHETIC_DATA_DIR
        / "X_synthetic.npy",

    "labels":
        SESSION_SYNTHETIC_DATA_DIR
        / "y_synthetic.npy",

    "metadata":
        SESSION_SYNTHETIC_DATA_DIR
        / "metadata_synthetic.csv",

    "manifest":
        SESSION_SYNTHETIC_DATA_DIR
        / "generation_manifest.json"
}

existing_synthetic_artifacts = {
    artifact_name:
        path.exists()
    for artifact_name, path
    in session_synthetic_files.items()
}

if any(
    existing_synthetic_artifacts.values()
) and not all(
    existing_synthetic_artifacts.values()
):
    raise RuntimeError(
        "Incomplete existing synthetic "
        "artifacts: "
        + str(
            existing_synthetic_artifacts
        )
    )

if all(
    existing_synthetic_artifacts.values()
):
    X_synthetic_saved = np.load(
        session_synthetic_files[
            "features"
        ],
        mmap_mode="r"
    )

    y_synthetic_saved = np.load(
        session_synthetic_files[
            "labels"
        ],
        mmap_mode="r"
    )

    metadata_synthetic_saved = (
        pd.read_csv(
            session_synthetic_files[
                "metadata"
            ]
        )
    )

    with open(
        session_synthetic_files[
            "manifest"
        ],
        "r",
        encoding="utf-8"
    ) as file:
        generation_manifest = (
            json.load(file)
        )

    assert np.array_equal(
        np.asarray(
            X_synthetic_saved
        ),
        X_synthetic_session
    )

    assert np.array_equal(
        np.asarray(
            y_synthetic_saved
        ),
        y_synthetic_session
    )

    assert np.array_equal(
        metadata_synthetic_saved[
            "parent_training_array_index"
        ].to_numpy(
            dtype=np.int64
        ),
        metadata_synthetic_session[
            "parent_training_array_index"
        ].to_numpy(
            dtype=np.int64
        )
    )

    for file_name, file_hash in (
        generation_manifest[
            "file_sha256"
        ].items()
    ):
        assert (
            calculate_sha256(
                SESSION_SYNTHETIC_DATA_DIR
                / file_name
            )
            == file_hash
        )

    print(
        "Existing synthetic dataset was "
        "reloaded and fully verified."
    )

else:
    np.save(
        session_synthetic_files[
            "features"
        ],
        X_synthetic_session
    )

    np.save(
        session_synthetic_files[
            "labels"
        ],
        y_synthetic_session
    )

    metadata_synthetic_session.to_csv(
        session_synthetic_files[
            "metadata"
        ],
        index=False
    )

    generation_manifest = {
        "experiment":
            "session_independent_"
            "synthetic_augmentation",

        "generation_method":
            "controlled_signal_domain_"
            "augmentation_v1",

        "random_seed":
            SYNTHETIC_GENERATION_SEED,

        "synthetic_to_real_ratio":
            SYNTHETIC_TO_REAL_RATIO,

        "source_partition":
            "session_independent_training",

        "source_samples":
            int(
                len(X_real_source)
            ),

        "source_sessions":
            int(
                len(source_session_ids)
            ),

        "synthetic_samples":
            int(
                len(X_synthetic_session)
            ),

        "synthetic_birds":
            int(
                np.sum(
                    y_synthetic_session
                    == 0
                )
            ),

        "synthetic_drones":
            int(
                np.sum(
                    y_synthetic_session
                    == 1
                )
            ),

        "tensor_shape":
            list(
                X_synthetic_session.shape
            ),

        "data_leakage_protection": {
            "parents_from_training_sessions_only":
                True,
            "validation_sessions_used":
                False,
            "test_sessions_used":
                False
        }
    }

    generation_manifest[
        "file_sha256"
    ] = {
        "X_synthetic.npy":
            calculate_sha256(
                session_synthetic_files[
                    "features"
                ]
            ),

        "y_synthetic.npy":
            calculate_sha256(
                session_synthetic_files[
                    "labels"
                ]
            ),

        "metadata_synthetic.csv":
            calculate_sha256(
                session_synthetic_files[
                    "metadata"
                ]
            )
    }

    with open(
        session_synthetic_files[
            "manifest"
        ],
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            generation_manifest,
            file,
            indent=2
        )

    print(
        "Session-independent synthetic "
        "dataset saved:",
        SESSION_SYNTHETIC_DATA_DIR.resolve()
    )

print(
    "Synthetic dataset persistence "
    "checks passed."
)

## 6. Paired Multi-Seed Classification Experiment

Two training configurations are compared using the same session-independent validation and test partitions:

1. **10% real-only:** 1,248 balanced real observations.
2. **10% real + synthetic 1:1:** 1,248 real and 1,248 synthetic observations.

Both configurations use:

- the fixed 29,121-parameter CNN;
- seeds 42, 52, 62, 72, and 82;
- identical optimizer and callback settings;
- validation-only threshold optimization;
- the same unseen-session test partition.

Each model seed is paired across the two configurations. Training remains protected by default and is activated only after artifact inspection.

In [ ]:
import random

import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from tensorflow.keras import (
    layers,
    models,
    regularizers
)

MODEL_SEEDS = [
    42,
    52,
    62,
    72,
    82
]

BATCH_SIZE = 64
MAX_EPOCHS = 50

SESSION_EXPERIMENT_OUTPUT_DIR = Path(
    "../outputs/"
    "session_independent_generalization"
)

SESSION_EXPERIMENT_CHECKPOINT_DIR = Path(
    "../checkpoints/"
    "session_independent_generalization"
)

SESSION_EXPERIMENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SESSION_EXPERIMENT_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Safe default. Never leave this True
# in the saved notebook.
RUN_SESSION_MODEL_TRAINING = False

print(
    "TensorFlow version:",
    tf.__version__
)

print(
    "Available GPUs:",
    tf.config.list_physical_devices(
        "GPU"
    )
)

print(
    "Model seeds:",
    MODEL_SEEDS
)

print(
    "Run session-model training:",
    RUN_SESSION_MODEL_TRAINING
)

In [ ]:
X_real_training = np.asarray(
    np.load(
        SESSION_PROCESSED_DATA_DIR
        / "X_train.npy",
        mmap_mode="r"
    )[
        real_subset_indices
    ],
    dtype=np.float32
)

y_real_training = np.asarray(
    np.load(
        SESSION_PROCESSED_DATA_DIR
        / "y_train.npy",
        mmap_mode="r"
    )[
        real_subset_indices
    ],
    dtype=np.uint8
)

X_synthetic_training = np.asarray(
    np.load(
        session_synthetic_files[
            "features"
        ],
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_synthetic_training = np.asarray(
    np.load(
        session_synthetic_files[
            "labels"
        ],
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_session_validation = np.asarray(
    np.load(
        SESSION_PROCESSED_DATA_DIR
        / "X_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_session_validation = np.asarray(
    np.load(
        SESSION_PROCESSED_DATA_DIR
        / "y_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_session_test = np.asarray(
    np.load(
        SESSION_PROCESSED_DATA_DIR
        / "X_test.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_session_test = np.asarray(
    np.load(
        SESSION_PROCESSED_DATA_DIR
        / "y_test.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

metadata_session_validation = (
    pd.read_csv(
        SESSION_PROCESSED_DATA_DIR
        / "metadata_validation.csv"
    )
)

metadata_session_test = pd.read_csv(
    SESSION_PROCESSED_DATA_DIR
    / "metadata_test.csv"
)

X_augmented_training = np.concatenate(
    [
        X_real_training,
        X_synthetic_training
    ],
    axis=0
)

y_augmented_training = np.concatenate(
    [
        y_real_training,
        y_synthetic_training
    ],
    axis=0
)

training_configurations = {
    "real_only": {
        "display_name":
            "10% session-independent "
            "real-only",

        "X":
            X_real_training,

        "y":
            y_real_training
    },

    "real_plus_synthetic": {
        "display_name":
            "10% session-independent real "
            "+ synthetic 1:1",

        "X":
            X_augmented_training,

        "y":
            y_augmented_training
    }
}

assert X_real_training.shape == (
    1248,
    5,
    150
)

assert X_synthetic_training.shape == (
    1248,
    5,
    150
)

assert X_augmented_training.shape == (
    2496,
    5,
    150
)

assert np.array_equal(
    np.bincount(
        y_real_training,
        minlength=2
    ),
    [
        624,
        624
    ]
)

assert np.array_equal(
    np.bincount(
        y_synthetic_training,
        minlength=2
    ),
    [
        624,
        624
    ]
)

assert np.array_equal(
    np.bincount(
        y_augmented_training,
        minlength=2
    ),
    [
        1248,
        1248
    ]
)

assert X_session_validation.shape == (
    6943,
    5,
    150
)

assert X_session_test.shape == (
    6539,
    5,
    150
)

print(
    "Real-only training tensor:",
    X_real_training.shape
)

print(
    "Augmented training tensor:",
    X_augmented_training.shape
)

print(
    "Validation tensor:",
    X_session_validation.shape
)

print(
    "Test tensor:",
    X_session_test.shape
)

for configuration_name, configuration in (
    training_configurations.items()
):
    print(
        configuration_name,
        "class counts:",
        np.bincount(
            configuration["y"],
            minlength=2
        )
    )

In [ ]:
def set_model_seed(
    model_seed
):
    tf.keras.backend.clear_session()

    random.seed(
        model_seed
    )

    np.random.seed(
        model_seed
    )

    tf.random.set_seed(
        model_seed
    )

    try:
        (
            tf.config.experimental
            .enable_op_determinism()
        )
    except Exception:
        pass


def build_model_datasets(
    configuration_name,
    model_seed
):
    configuration = (
        training_configurations[
            configuration_name
        ]
    )

    X_training_model = (
        configuration["X"][
            ...,
            np.newaxis
        ]
    )

    X_validation_model = (
        X_session_validation[
            ...,
            np.newaxis
        ]
    )

    X_test_model = (
        X_session_test[
            ...,
            np.newaxis
        ]
    )

    training_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_training_model,
            configuration["y"]
        ))
        .shuffle(
            buffer_size=len(
                configuration["y"]
            ),
            seed=model_seed,
            reshuffle_each_iteration=True
        )
        .batch(BATCH_SIZE)
        .prefetch(
            tf.data.AUTOTUNE
        )
    )

    validation_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_validation_model,
            y_session_validation
        ))
        .batch(BATCH_SIZE)
        .prefetch(
            tf.data.AUTOTUNE
        )
    )

    test_dataset = (
        tf.data.Dataset
        .from_tensor_slices((
            X_test_model,
            y_session_test
        ))
        .batch(BATCH_SIZE)
        .prefetch(
            tf.data.AUTOTUNE
        )
    )

    return (
        training_dataset,
        validation_dataset,
        test_dataset
    )


def build_fixed_cnn(
    input_shape=(5, 150, 1)
):
    model = models.Sequential([
        layers.Input(
            shape=input_shape
        ),

        layers.Conv2D(
            16,
            kernel_size=(3, 7),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            32,
            kernel_size=(3, 5),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            64,
            kernel_size=(3, 3),
            padding="same",
            use_bias=True
        ),

        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            32,
            activation="relu",
            kernel_regularizer=(
                regularizers.l2(
                    1e-4
                )
            )
        ),

        layers.Dropout(0.30),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=(
            tf.keras.optimizers.Adam(
                learning_rate=1e-3
            )
        ),

        loss="binary_crossentropy",

        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.AUC(
                name="roc_auc"
            ),

            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            ),

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            )
        ]
    )

    return model


set_model_seed(
    MODEL_SEEDS[0]
)

verification_model = build_fixed_cnn()

assert (
    verification_model.count_params()
    == 29121
)

print(
    "Architecture parameters:",
    verification_model.count_params()
)

print(
    "Architecture verified against "
    "the official-split experiments."
)

del verification_model

tf.keras.backend.clear_session()

In [ ]:
def get_session_run_paths(
    configuration_name,
    model_seed
):
    run_name = (
        f"{configuration_name}_"
        f"seed_{model_seed}"
    )

    output_directory = (
        SESSION_EXPERIMENT_OUTPUT_DIR
        / run_name
    )

    return {
        "run_name":
            run_name,

        "output_directory":
            output_directory,

        "checkpoint":
            SESSION_EXPERIMENT_CHECKPOINT_DIR
            / f"{run_name}.keras",

        "history":
            output_directory
            / "training_history.csv",

        "training_log":
            output_directory
            / "training_log.csv",

        "configuration":
            output_directory
            / "run_configuration.json",

        "validation_results":
            output_directory
            / "validation_results.csv",

        "threshold_search":
            output_directory
            / "validation_threshold_search.csv",

        "test_metrics":
            output_directory
            / "test_metrics.csv",

        "test_predictions":
            output_directory
            / "test_predictions.csv"
    }


session_run_status_records = []

for configuration_name in (
    training_configurations
):
    for model_seed in MODEL_SEEDS:
        run_paths = get_session_run_paths(
            configuration_name,
            model_seed
        )

        checkpoint_exists = (
            run_paths[
                "checkpoint"
            ].exists()
        )

        history_exists = (
            run_paths[
                "history"
            ].exists()
        )

        validation_exists = (
            run_paths[
                "validation_results"
            ].exists()
        )

        threshold_exists = (
            run_paths[
                "threshold_search"
            ].exists()
        )

        test_metrics_exists = (
            run_paths[
                "test_metrics"
            ].exists()
        )

        test_predictions_exist = (
            run_paths[
                "test_predictions"
            ].exists()
        )

        training_complete = (
            checkpoint_exists
            and history_exists
        )

        validation_complete = (
            validation_exists
            and threshold_exists
        )

        test_complete = (
            test_metrics_exists
            and test_predictions_exist
        )

        inconsistent_artifacts = (
            checkpoint_exists
            != history_exists
        ) or (
            validation_exists
            != threshold_exists
        ) or (
            test_metrics_exists
            != test_predictions_exist
        ) or (
            validation_complete
            and not training_complete
        ) or (
            test_complete
            and not validation_complete
        )

        if inconsistent_artifacts:
            status = (
                "inconsistent_artifacts"
            )

        elif (
            training_complete
            and validation_complete
            and test_complete
        ):
            status = "complete"

        elif (
            training_complete
            and validation_complete
        ):
            status = (
                "validated_awaiting_test"
            )

        elif training_complete:
            status = (
                "trained_awaiting_validation"
            )

        else:
            status = "not_started"

        session_run_status_records.append({
            "configuration":
                configuration_name,

            "seed":
                model_seed,

            "status":
                status,

            "checkpoint_exists":
                checkpoint_exists,

            "history_exists":
                history_exists,

            "validation_exists":
                validation_exists,

            "test_exists":
                test_complete,

            "training_complete":
                training_complete,

            "run_complete":
                (
                    training_complete
                    and validation_complete
                    and test_complete
                ),

            "output_directory":
                str(
                    run_paths[
                        "output_directory"
                    ]
                )
        })

session_run_status_df = pd.DataFrame(
    session_run_status_records
)

display(
    session_run_status_df
)

inconsistent_session_runs = (
    session_run_status_df[
        session_run_status_df[
            "status"
        ] == "inconsistent_artifacts"
    ]
)

if not (
    inconsistent_session_runs.empty
):
    raise RuntimeError(
        "Inconsistent session-experiment "
        "artifacts were found."
    )

if RUN_SESSION_MODEL_TRAINING:
    print(
        "Training mode requested."
    )
else:
    print(
        "SAFE INSPECTION MODE: no new "
        "session-model training is "
        "authorized."
    )

## 7. Protected Multi-Seed Training

Ten paired models are trained:

- five real-only models;
- five real-plus-synthetic models.

Every run uses an independent checkpoint and output directory. Completed runs are skipped automatically, while partial or inconsistent artifacts stop execution to prevent accidental overwriting.

In [ ]:
def train_session_run(
    configuration_name,
    model_seed
):
    run_paths = get_session_run_paths(
        configuration_name,
        model_seed
    )

    checkpoint_exists = (
        run_paths[
            "checkpoint"
        ].exists()
    )

    history_exists = (
        run_paths[
            "history"
        ].exists()
    )

    if (
        checkpoint_exists
        and history_exists
    ):
        print(
            "Training already complete; "
            "skipping:",
            run_paths["run_name"]
        )

        return {
            "configuration":
                configuration_name,

            "seed":
                model_seed,

            "status":
                "already_trained"
        }

    if (
        checkpoint_exists
        != history_exists
    ):
        raise RuntimeError(
            "Inconsistent training "
            "artifacts for "
            + run_paths["run_name"]
        )

    unexpected_existing_files = [
        path
        for path in [
            run_paths[
                "training_log"
            ],
            run_paths[
                "configuration"
            ],
            run_paths[
                "validation_results"
            ],
            run_paths[
                "threshold_search"
            ],
            run_paths[
                "test_metrics"
            ],
            run_paths[
                "test_predictions"
            ]
        ]
        if path.exists()
    ]

    if unexpected_existing_files:
        raise FileExistsError(
            "Unexpected existing artifacts "
            "for "
            + run_paths["run_name"]
            + ":\n"
            + "\n".join(
                str(path.resolve())
                for path
                in unexpected_existing_files
            )
        )

    run_paths[
        "output_directory"
    ].mkdir(
        parents=True,
        exist_ok=True
    )

    set_model_seed(
        model_seed
    )

    (
        training_dataset,
        validation_dataset,
        _
    ) = build_model_datasets(
        configuration_name,
        model_seed
    )

    model = build_fixed_cnn()

    assert (
        model.count_params()
        == 29121
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=run_paths[
                "checkpoint"
            ],
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            run_paths[
                "training_log"
            ]
        )
    ]

    training_observations = int(
        len(
            training_configurations[
                configuration_name
            ]["y"]
        )
    )

    print()
    print("=" * 72)

    print(
        "Training:",
        run_paths["run_name"]
    )

    print(
        "Training observations:",
        training_observations
    )

    print("=" * 72)

    history = model.fit(
        training_dataset,
        validation_data=(
            validation_dataset
        ),
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    history_df = pd.DataFrame(
        history.history
    )

    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(history_df) + 1
        )
    )

    history_df.to_csv(
        run_paths[
            "history"
        ],
        index=False
    )

    best_history_index = (
        history_df[
            "val_loss"
        ].idxmin()
    )

    best_epoch = int(
        history_df.loc[
            best_history_index,
            "epoch"
        ]
    )

    best_validation_loss = float(
        history_df.loc[
            best_history_index,
            "val_loss"
        ]
    )

    run_configuration = {
        "experiment":
            "session_independent_"
            "generalization",

        "configuration":
            configuration_name,

        "configuration_display_name":
            training_configurations[
                configuration_name
            ]["display_name"],

        "model_seed":
            model_seed,

        "split_seed":
            RANDOM_SEED,

        "training_observations":
            training_observations,

        "real_training_observations":
            int(
                len(
                    y_real_training
                )
            ),

        "synthetic_training_observations":
            (
                0
                if configuration_name
                == "real_only"
                else int(
                    len(
                        y_synthetic_training
                    )
                )
            ),

        "validation_observations":
            int(
                len(
                    y_session_validation
                )
            ),

        "test_observations":
            int(
                len(
                    y_session_test
                )
            ),

        "training_sessions":
            80,

        "validation_sessions":
            10,

        "test_sessions":
            10,

        "batch_size":
            BATCH_SIZE,

        "maximum_epochs":
            MAX_EPOCHS,

        "completed_epochs":
            int(
                len(history_df)
            ),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss,

        "model_parameters":
            int(
                model.count_params()
            ),

        "session_assignment_file":
            str(
                SESSION_ASSIGNMENT_PATH
            )
    }

    with open(
        run_paths[
            "configuration"
        ],
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            run_configuration,
            file,
            indent=2
        )

    assert (
        run_paths[
            "checkpoint"
        ].exists()
    )

    assert (
        run_paths[
            "history"
        ].exists()
    )

    print(
        "Completed:",
        run_paths["run_name"]
    )

    print(
        "Completed epochs:",
        len(history_df)
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best validation loss:",
        best_validation_loss
    )

    result = {
        "configuration":
            configuration_name,

        "seed":
            model_seed,

        "status":
            "trained",

        "completed_epochs":
            int(
                len(history_df)
            ),

        "best_epoch":
            best_epoch,

        "best_validation_loss":
            best_validation_loss
    }

    del model
    del history

    tf.keras.backend.clear_session()

    gc.collect()

    return result

In [ ]:
pending_session_training_records = []

for model_seed in MODEL_SEEDS:
    for configuration_name in [
        "real_only",
        "real_plus_synthetic"
    ]:
        run_paths = (
            get_session_run_paths(
                configuration_name,
                model_seed
            )
        )

        checkpoint_exists = (
            run_paths[
                "checkpoint"
            ].exists()
        )

        history_exists = (
            run_paths[
                "history"
            ].exists()
        )

        if (
            checkpoint_exists
            != history_exists
        ):
            raise RuntimeError(
                "Inconsistent training "
                "artifacts for "
                + run_paths["run_name"]
            )

        training_complete = (
            checkpoint_exists
            and history_exists
        )

        pending_session_training_records.append({
            "configuration":
                configuration_name,

            "seed":
                model_seed,

            "training_observations":
                len(
                    training_configurations[
                        configuration_name
                    ]["y"]
                ),

            "training_complete":
                training_complete,

            "action":
                (
                    "skip"
                    if training_complete
                    else "train"
                )
        })

pending_session_training_df = (
    pd.DataFrame(
        pending_session_training_records
    )
)

display(
    pending_session_training_df
)

print(
    "Runs requiring training:",
    int(
        (
            pending_session_training_df[
                "action"
            ] == "train"
        ).sum()
    )
)

In [ ]:
# Final safety lock. Completed artifacts are reloaded;
# no new model training is authorized.
RUN_SESSION_MODEL_TRAINING = False

print(
    "SAFE MODE: session-model training "
    "is disabled."
)

In [ ]:
if not RUN_SESSION_MODEL_TRAINING:
    print(
        "Session-model training remains "
        "disabled."
    )

else:
    session_training_records = []

    for model_seed in MODEL_SEEDS:
        for configuration_name in [
            "real_only",
            "real_plus_synthetic"
        ]:
            training_result = (
                train_session_run(
                    configuration_name,
                    model_seed
                )
            )

            session_training_records.append(
                training_result
            )

    session_training_summary_df = (
        pd.DataFrame(
            session_training_records
        )
    )

    display(
        session_training_summary_df
    )

    RUN_SESSION_MODEL_TRAINING = False

    print(
        "All requested session-model "
        "training calls finished."
    )

    print(
        "RUN_SESSION_MODEL_TRAINING was "
        "reset to False in memory."
    )

## 8. Validation Threshold Selection and Test Evaluation

For each trained model:

1. probabilities are generated for the session-independent validation partition;
2. the classification threshold is selected by maximizing validation macro-F1;
3. balanced accuracy and accuracy are used as tie-breakers;
4. the selected threshold is locked;
5. the model is evaluated once on the unseen-session test partition.

No test metric participates in threshold selection.

In [ ]:
from sklearn.metrics import (
    confusion_matrix
)


def calculate_binary_metrics(
    true_labels,
    probabilities,
    threshold
):
    predicted_labels = (
        probabilities
        >= threshold
    ).astype(np.uint8)

    confusion = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=[
            0,
            1
        ]
    )

    true_birds = int(
        confusion[0, 0]
    )

    birds_predicted_as_drones = int(
        confusion[0, 1]
    )

    drones_predicted_as_birds = int(
        confusion[1, 0]
    )

    true_drones = int(
        confusion[1, 1]
    )

    return {
        "accuracy":
            float(
                accuracy_score(
                    true_labels,
                    predicted_labels
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    true_labels,
                    predicted_labels
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    true_labels,
                    predicted_labels,
                    average="macro",
                    zero_division=0
                )
            ),

        "bird_precision":
            float(
                precision_score(
                    true_labels,
                    predicted_labels,
                    pos_label=0,
                    zero_division=0
                )
            ),

        "bird_recall":
            float(
                recall_score(
                    true_labels,
                    predicted_labels,
                    pos_label=0,
                    zero_division=0
                )
            ),

        "bird_f1":
            float(
                f1_score(
                    true_labels,
                    predicted_labels,
                    pos_label=0,
                    zero_division=0
                )
            ),

        "drone_precision":
            float(
                precision_score(
                    true_labels,
                    predicted_labels,
                    pos_label=1,
                    zero_division=0
                )
            ),

        "drone_recall":
            float(
                recall_score(
                    true_labels,
                    predicted_labels,
                    pos_label=1,
                    zero_division=0
                )
            ),

        "drone_f1":
            float(
                f1_score(
                    true_labels,
                    predicted_labels,
                    pos_label=1,
                    zero_division=0
                )
            ),

        "roc_auc":
            float(
                roc_auc_score(
                    true_labels,
                    probabilities
                )
            ),

        "true_birds":
            true_birds,

        "birds_predicted_as_drones":
            birds_predicted_as_drones,

        "drones_predicted_as_birds":
            drones_predicted_as_birds,

        "true_drones":
            true_drones
    }

In [ ]:
def evaluate_session_run(
    configuration_name,
    model_seed
):
    run_paths = get_session_run_paths(
        configuration_name,
        model_seed
    )

    required_training_files = [
        run_paths["checkpoint"],
        run_paths["history"],
        run_paths["configuration"]
    ]

    missing_training_files = [
        path
        for path in required_training_files
        if not path.exists()
    ]

    if missing_training_files:
        raise FileNotFoundError(
            "Training is incomplete for "
            + run_paths["run_name"]
            + ":\n"
            + "\n".join(
                str(path.resolve())
                for path
                in missing_training_files
            )
        )

    evaluation_paths = [
        run_paths[
            "validation_results"
        ],
        run_paths[
            "threshold_search"
        ],
        run_paths[
            "test_metrics"
        ],
        run_paths[
            "test_predictions"
        ]
    ]

    evaluation_exists = [
        path.exists()
        for path in evaluation_paths
    ]

    if any(
        evaluation_exists
    ) and not all(
        evaluation_exists
    ):
        raise RuntimeError(
            "Incomplete evaluation artifacts "
            "for "
            + run_paths["run_name"]
        )

    if all(
        evaluation_exists
    ):
        saved_test_metrics = (
            pd.read_csv(
                run_paths[
                    "test_metrics"
                ]
            )
        )

        assert len(
            saved_test_metrics
        ) == 1

        print(
            "Evaluation already complete; "
            "reusing:",
            run_paths["run_name"]
        )

        return (
            saved_test_metrics
            .iloc[0]
            .to_dict()
        )

    with open(
        run_paths[
            "configuration"
        ],
        "r",
        encoding="utf-8"
    ) as file:
        run_configuration = (
            json.load(file)
        )

    set_model_seed(
        model_seed
    )

    model = tf.keras.models.load_model(
        run_paths[
            "checkpoint"
        ]
    )

    assert (
        model.count_params()
        == 29121
    )

    (
        _,
        validation_dataset,
        test_dataset
    ) = build_model_datasets(
        configuration_name,
        model_seed
    )

    validation_probabilities = (
        model.predict(
            validation_dataset,
            verbose=0
        )
        .reshape(-1)
    )

    assert (
        validation_probabilities.shape
        == y_session_validation.shape
    )

    assert np.isfinite(
        validation_probabilities
    ).all()

    threshold_records = []

    for threshold in np.linspace(
        0.01,
        0.99,
        199
    ):
        threshold_metrics = (
            calculate_binary_metrics(
                y_session_validation,
                validation_probabilities,
                float(threshold)
            )
        )

        threshold_records.append({
            "threshold":
                float(threshold),

            **threshold_metrics
        })

    threshold_search_df = (
        pd.DataFrame(
            threshold_records
        )
    )

    optimal_validation_row = (
        threshold_search_df
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "accuracy"
            ],
            ascending=[
                False,
                False,
                False
            ],
            kind="mergesort"
        )
        .iloc[0]
    )

    locked_threshold = float(
        optimal_validation_row[
            "threshold"
        ]
    )

    validation_result = {
        "configuration":
            configuration_name,

        "configuration_display_name":
            training_configurations[
                configuration_name
            ]["display_name"],

        "seed":
            model_seed,

        "threshold":
            locked_threshold,

        **{
            metric_name:
                optimal_validation_row[
                    metric_name
                ]
            for metric_name in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "bird_precision",
                "bird_recall",
                "bird_f1",
                "drone_precision",
                "drone_recall",
                "drone_f1",
                "roc_auc",
                "true_birds",
                "birds_predicted_as_drones",
                "drones_predicted_as_birds",
                "true_drones"
            ]
        }
    }

    pd.DataFrame([
        validation_result
    ]).to_csv(
        run_paths[
            "validation_results"
        ],
        index=False
    )

    threshold_search_df.to_csv(
        run_paths[
            "threshold_search"
        ],
        index=False
    )

    test_probabilities = (
        model.predict(
            test_dataset,
            verbose=0
        )
        .reshape(-1)
    )

    assert (
        test_probabilities.shape
        == y_session_test.shape
    )

    assert np.isfinite(
        test_probabilities
    ).all()

    test_metrics = (
        calculate_binary_metrics(
            y_session_test,
            test_probabilities,
            locked_threshold
        )
    )

    test_predictions = (
        test_probabilities
        >= locked_threshold
    ).astype(np.uint8)

    test_result = {
        "configuration":
            configuration_name,

        "configuration_display_name":
            training_configurations[
                configuration_name
            ]["display_name"],

        "seed":
            model_seed,

        "training_observations":
            int(
                len(
                    training_configurations[
                        configuration_name
                    ]["y"]
                )
            ),

        "best_epoch":
            int(
                run_configuration[
                    "best_epoch"
                ]
            ),

        "best_validation_loss":
            float(
                run_configuration[
                    "best_validation_loss"
                ]
            ),

        "threshold":
            locked_threshold,

        **test_metrics
    }

    pd.DataFrame([
        test_result
    ]).to_csv(
        run_paths[
            "test_metrics"
        ],
        index=False
    )

    test_predictions_df = (
        metadata_session_test.copy()
    )

    test_predictions_df[
        "true_label"
    ] = y_session_test

    test_predictions_df[
        "drone_probability"
    ] = test_probabilities

    test_predictions_df[
        "predicted_label"
    ] = test_predictions

    test_predictions_df[
        "prediction_correct"
    ] = (
        test_predictions
        == y_session_test
    )

    test_predictions_df[
        "locked_threshold"
    ] = locked_threshold

    test_predictions_df.to_csv(
        run_paths[
            "test_predictions"
        ],
        index=False
    )

    print()
    print(
        "Evaluated:",
        run_paths["run_name"]
    )

    print(
        "Locked threshold:",
        round(
            locked_threshold,
            4
        )
    )

    print(
        "Test macro-F1:",
        round(
            test_metrics[
                "macro_f1"
            ],
            4
        )
    )

    print(
        "Test balanced accuracy:",
        round(
            test_metrics[
                "balanced_accuracy"
            ],
            4
        )
    )

    del model

    tf.keras.backend.clear_session()

    gc.collect()

    return test_result

In [ ]:
assert (
    RUN_SESSION_MODEL_TRAINING
    is False
)

session_test_metric_records = []

for model_seed in MODEL_SEEDS:
    for configuration_name in [
        "real_only",
        "real_plus_synthetic"
    ]:
        test_result = (
            evaluate_session_run(
                configuration_name,
                model_seed
            )
        )

        session_test_metric_records.append(
            test_result
        )

session_test_metrics_df = (
    pd.DataFrame(
        session_test_metric_records
    )
    .sort_values([
        "seed",
        "configuration"
    ])
    .reset_index(drop=True)
)

session_test_metrics_df.to_csv(
    SESSION_EXPERIMENT_OUTPUT_DIR
    / "all_seed_test_metrics.csv",
    index=False
)

display_columns = [
    "configuration_display_name",
    "seed",
    "best_epoch",
    "best_validation_loss",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

display(
    session_test_metrics_df[
        display_columns
    ].style.format({
        "best_validation_loss":
            "{:.4f}",
        "threshold":
            "{:.4f}",
        "accuracy":
            "{:.4f}",
        "balanced_accuracy":
            "{:.4f}",
        "macro_f1":
            "{:.4f}",
        "bird_precision":
            "{:.4f}",
        "bird_recall":
            "{:.4f}",
        "bird_f1":
            "{:.4f}",
        "drone_recall":
            "{:.4f}",
        "roc_auc":
            "{:.4f}"
    })
)

print(
    "All ten locked-threshold test "
    "evaluations were completed or "
    "safely reloaded."
)

## 9. Multi-Seed Robustness Analysis

The real-only and augmented configurations are compared by paired training seed.

The analysis reports:

- mean and standard deviation;
- minimum and maximum performance;
- paired seed-level improvement;
- augmentation wins, ties, and losses;
- descriptive 95% confidence intervals for the mean paired improvement.

Because only five seeds are available, confidence intervals are descriptive and depend on the normality assumption for paired differences.

In [ ]:
SESSION_SUMMARY_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

session_configuration_summary_df = (
    session_test_metrics_df
    .groupby(
        [
            "configuration",
            "configuration_display_name"
        ],
        observed=True
    )[SESSION_SUMMARY_METRICS]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
)

session_configuration_summary_df.columns = [
    f"{metric}_{statistic}"
    for metric, statistic
    in session_configuration_summary_df.columns
]

session_configuration_summary_df = (
    session_configuration_summary_df
    .reset_index()
)

session_configuration_summary_df.to_csv(
    SESSION_EXPERIMENT_OUTPUT_DIR
    / "configuration_summary.csv",
    index=False
)

display(
    session_configuration_summary_df
    .style
    .format({
        column: "{:.4f}"
        for column
        in session_configuration_summary_df.columns
        if column not in [
            "configuration",
            "configuration_display_name"
        ]
    })
)

In [ ]:
from scipy.stats import t


SESSION_METRIC_DISPLAY_NAMES = {
    "accuracy":
        "Accuracy",

    "balanced_accuracy":
        "Balanced accuracy",

    "macro_f1":
        "Macro-F1",

    "bird_precision":
        "Bird precision",

    "bird_recall":
        "Bird recall",

    "bird_f1":
        "Bird F1",

    "drone_recall":
        "Drone recall",

    "roc_auc":
        "ROC-AUC"
}

paired_improvement_records = []
paired_seed_records = []

for (
    metric_name,
    metric_display_name
) in SESSION_METRIC_DISPLAY_NAMES.items():
    metric_pivot = (
        session_test_metrics_df
        .pivot(
            index="seed",
            columns="configuration",
            values=metric_name
        )
        .reindex(
            MODEL_SEEDS
        )
    )

    assert not (
        metric_pivot.isna().any().any()
    )

    paired_difference = (
        metric_pivot[
            "real_plus_synthetic"
        ]
        - metric_pivot[
            "real_only"
        ]
    )

    mean_improvement = float(
        paired_difference.mean()
    )

    improvement_standard_deviation = float(
        paired_difference.std(
            ddof=1
        )
    )

    improvement_standard_error = (
        improvement_standard_deviation
        / np.sqrt(
            len(
                paired_difference
            )
        )
    )

    critical_t = float(
        t.ppf(
            0.975,
            df=(
                len(
                    paired_difference
                )
                - 1
            )
        )
    )

    confidence_interval_lower = (
        mean_improvement
        - critical_t
        * improvement_standard_error
    )

    confidence_interval_upper = (
        mean_improvement
        + critical_t
        * improvement_standard_error
    )

    paired_improvement_records.append({
        "metric":
            metric_name,

        "metric_display_name":
            metric_display_name,

        "seeds":
            len(
                paired_difference
            ),

        "real_only_mean":
            float(
                metric_pivot[
                    "real_only"
                ].mean()
            ),

        "augmented_mean":
            float(
                metric_pivot[
                    "real_plus_synthetic"
                ].mean()
            ),

        "mean_paired_improvement":
            mean_improvement,

        "improvement_standard_deviation":
            improvement_standard_deviation,

        "confidence_interval_95_lower":
            confidence_interval_lower,

        "confidence_interval_95_upper":
            confidence_interval_upper,

        "augmented_wins":
            int(
                np.sum(
                    paired_difference
                    > 0
                )
            ),

        "ties":
            int(
                np.sum(
                    np.isclose(
                        paired_difference,
                        0.0
                    )
                )
            ),

        "augmented_losses":
            int(
                np.sum(
                    paired_difference
                    < 0
                )
            )
    })

    for model_seed in MODEL_SEEDS:
        paired_seed_records.append({
            "seed":
                model_seed,

            "metric":
                metric_name,

            "real_only":
                float(
                    metric_pivot.loc[
                        model_seed,
                        "real_only"
                    ]
                ),

            "real_plus_synthetic":
                float(
                    metric_pivot.loc[
                        model_seed,
                        "real_plus_synthetic"
                    ]
                ),

            "paired_improvement":
                float(
                    paired_difference.loc[
                        model_seed
                    ]
                )
        })

session_paired_summary_df = (
    pd.DataFrame(
        paired_improvement_records
    )
)

session_paired_seed_df = pd.DataFrame(
    paired_seed_records
)

session_paired_summary_df.to_csv(
    SESSION_EXPERIMENT_OUTPUT_DIR
    / "paired_improvement_summary.csv",
    index=False
)

session_paired_seed_df.to_csv(
    SESSION_EXPERIMENT_OUTPUT_DIR
    / "paired_seed_improvements.csv",
    index=False
)

display(
    session_paired_summary_df
    .style
    .format({
        "real_only_mean":
            "{:.4f}",
        "augmented_mean":
            "{:.4f}",
        "mean_paired_improvement":
            "{:+.4f}",
        "improvement_standard_deviation":
            "{:.4f}",
        "confidence_interval_95_lower":
            "{:+.4f}",
        "confidence_interval_95_upper":
            "{:+.4f}"
    })
)

In [ ]:
import matplotlib.pyplot as plt

selected_plot_metrics = [
    (
        "balanced_accuracy",
        "Balanced Accuracy"
    ),
    (
        "macro_f1",
        "Macro-F1"
    ),
    (
        "bird_f1",
        "Bird F1"
    ),
    (
        "roc_auc",
        "ROC-AUC"
    )
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10),
    constrained_layout=True
)

x_positions = [
    0,
    1
]

x_labels = [
    "Real-only",
    "Real +\nsynthetic 1:1"
]

for axis, (
    metric_name,
    metric_title
) in zip(
    axes.flat,
    selected_plot_metrics
):
    metric_pivot = (
        session_test_metrics_df
        .pivot(
            index="seed",
            columns="configuration",
            values=metric_name
        )
        .reindex(
            MODEL_SEEDS
        )
    )

    for model_seed in MODEL_SEEDS:
        axis.plot(
            x_positions,
            [
                metric_pivot.loc[
                    model_seed,
                    "real_only"
                ],
                metric_pivot.loc[
                    model_seed,
                    "real_plus_synthetic"
                ]
            ],
            marker="o",
            linewidth=1.8,
            label=(
                f"Seed {model_seed}"
            )
        )

    axis.set_title(
        metric_title
    )

    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        x_labels
    )

    axis.set_ylim(
        0.0,
        1.02
    )

    axis.set_ylabel(
        "Unseen-session test score"
    )

    axis.grid(
        alpha=0.25
    )

axes[0, 0].legend(
    title="Training seed",
    loc="lower right"
)

fig.suptitle(
    "Paired Session-Independent Performance:\n"
    "Real-Only versus Synthetic-Augmented Training",
    fontsize=14
)

plt.show()

## 10. Descriptive Comparison with the Official Split

The locked five-seed aggregate results are compared with Notebook 09 to determine whether the augmentation effect remains similar after enforcing session independence. Because the two protocols use different test observations, this is a descriptive comparison of aggregate gains rather than a paired statistical comparison of absolute scores.

In [ ]:
# Descriptive comparison:
# official segment-level split versus
# session-independent split.
#
# No model training or test inference is performed.

from pathlib import Path

official_split_summary = {
    "accuracy": {
        "real_only": 0.8794,
        "real_plus_synthetic": 0.9598
    },
    "balanced_accuracy": {
        "real_only": 0.8518,
        "real_plus_synthetic": 0.9268
    },
    "macro_f1": {
        "real_only": 0.8003,
        "real_plus_synthetic": 0.9209
    },
    "bird_f1": {
        "real_only": 0.6749,
        "real_plus_synthetic": 0.8654
    },
    "drone_recall": {
        "real_only": 0.8903,
        "real_plus_synthetic": 0.9730
    },
    "roc_auc": {
        "real_only": 0.9276,
        "real_plus_synthetic": 0.9863
    }
}

session_independent_summary = {
    "accuracy": {
        "real_only": 0.9233,
        "real_plus_synthetic": 0.9851
    },
    "balanced_accuracy": {
        "real_only": 0.8941,
        "real_plus_synthetic": 0.9723
    },
    "macro_f1": {
        "real_only": 0.8485,
        "real_plus_synthetic": 0.9615
    },
    "bird_f1": {
        "real_only": 0.7427,
        "real_plus_synthetic": 0.9314
    },
    "drone_recall": {
        "real_only": 0.9310,
        "real_plus_synthetic": 0.9885
    },
    "roc_auc": {
        "real_only": 0.9606,
        "real_plus_synthetic": 0.9978
    }
}

metric_display_names = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced accuracy",
    "macro_f1": "Macro-F1",
    "bird_f1": "Bird F1",
    "drone_recall": "Drone recall",
    "roc_auc": "ROC-AUC"
}

comparison_rows = []

for metric_name in official_split_summary:
    official_real = (
        official_split_summary[
            metric_name
        ]["real_only"]
    )

    official_augmented = (
        official_split_summary[
            metric_name
        ]["real_plus_synthetic"]
    )

    session_real = (
        session_independent_summary[
            metric_name
        ]["real_only"]
    )

    session_augmented = (
        session_independent_summary[
            metric_name
        ]["real_plus_synthetic"]
    )

    comparison_rows.append({
        "metric":
            metric_name,

        "metric_display_name":
            metric_display_names[
                metric_name
            ],

        "official_real_only_mean":
            official_real,

        "official_augmented_mean":
            official_augmented,

        "official_augmentation_gain":
            official_augmented
            - official_real,

        "session_independent_real_only_mean":
            session_real,

        "session_independent_augmented_mean":
            session_augmented,

        "session_independent_augmentation_gain":
            session_augmented
            - session_real,

        "change_in_augmentation_gain":
            (
                session_augmented
                - session_real
            )
            - (
                official_augmented
                - official_real
            )
    })

split_comparison_df = pd.DataFrame(
    comparison_rows
)

display(
    split_comparison_df.style.format({
        "official_real_only_mean":
            "{:.4f}",

        "official_augmented_mean":
            "{:.4f}",

        "official_augmentation_gain":
            "{:+.4f}",

        "session_independent_real_only_mean":
            "{:.4f}",

        "session_independent_augmented_mean":
            "{:.4f}",

        "session_independent_augmentation_gain":
            "{:+.4f}",

        "change_in_augmentation_gain":
            "{:+.4f}"
    })
)

comparison_output_dir = Path(
    "../outputs/session_independent_generalization"
)

comparison_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

split_comparison_df.to_csv(
    comparison_output_dir
    / "official_vs_session_independent_comparison.csv",
    index=False
)

print(
    "Descriptive split comparison saved."
)

print(
    "Important: these values come from different "
    "test partitions and should not be interpreted "
    "as a controlled paired comparison."
)

## 11. Conclusion and Limitations

This experiment evaluated whether transformation-based synthetic augmentation
continues to improve radar bird–drone classification when recording sessions are
strictly separated between training, validation, and test partitions.

The session-independent protocol contained no session overlap. Synthetic examples
were generated exclusively from the balanced real training subset, while decision
thresholds were selected using only the validation partition. The unseen test
partition was used only after the complete evaluation protocol had been fixed.

Across five paired model seeds, adding synthetic data at a 1:1 synthetic-to-real
ratio improved mean test performance:

- Accuracy increased from **0.9233 to 0.9851**.
- Balanced accuracy increased from **0.8941 to 0.9723**.
- Macro-F1 increased from **0.8485 to 0.9615**.
- Bird F1 increased from **0.7427 to 0.9314**.
- Drone recall increased from **0.9310 to 0.9885**.
- ROC-AUC increased from **0.9606 to 0.9978**.

Synthetic augmentation also substantially reduced sensitivity to model
initialization. Macro-F1 standard deviation decreased from **0.1327 to 0.0102**,
while balanced-accuracy standard deviation decreased from **0.0837 to 0.0059**.
The worst observed macro-F1 improved from **0.6638 to 0.9490**.

The augmentation gains were broadly consistent with those previously observed
using the official segment-level split. In particular, the balanced-accuracy gain
changed from **+0.0750 to +0.0782**, and the bird-F1 gain changed from
**+0.1905 to +0.1887**. This indicates that the principal augmentation benefit
was preserved after preventing segments from the same recording session from
appearing across dataset partitions.

Overall, the results support the conclusion that transformation-based synthetic
augmentation improves both the effectiveness and robustness of low-data
micro-Doppler classification, including generalization to unseen recording
sessions.

### Limitations

Several limitations must be considered:

1. Only one session assignment was evaluated. The five repetitions vary model
   initialization and training order, but not the underlying session split.
2. The experiment contains only five paired model seeds. Consequently, the
   confidence intervals are wide and include zero; the results should therefore
   be interpreted as strong descriptive evidence rather than definitive
   statistical significance.
3. Singleton bird subtypes, pigeon and raven, were necessarily retained in the
   training partition and could not be evaluated on unseen sessions.
4. The test partition does not independently contain every drone model, although
   all six drone models are represented across validation and test together.
5. Synthetic samples are transformation-based derivatives of real training
   observations and do not reproduce every physical or environmental source of
   radar variability.
6. Results from the official and session-independent protocols use different test
   partitions. Their absolute scores are therefore compared descriptively and
   must not be treated as a controlled paired comparison.

A future extension should repeat the complete experiment using multiple valid
session assignments or grouped cross-validation to quantify variability caused
by the choice of held-out recording sessions.